# PyTorch Compilation Stack: TorchDynamo, Inductor, CUDA Graphs

**Model:** Qwen3.5-9B (9.0B params, 17.9 GB fp16)
**GPU:** NVIDIA RTX 5090 (32 GB GDDR7, 1.79 TB/s)

This notebook demonstrates each stage of the `torch.compile` pipeline on a real 9B-parameter model and measures the speedup from kernel fusion and CUDA graphs.

> **Note:** This notebook requires ~20 GB of VRAM and a local GPU. It will not run on a free Colab T4. The outputs below are cached from a run on an RTX 5090.

## Section 1: Eager baseline

Without `torch.compile`, PyTorch executes each operation (RMSNorm, linear, SiLU, attention) as a separate CUDA kernel. Each kernel reads its input from HBM and writes its output back.

In [ ]:
import torch, time, warnings
warnings.filterwarnings('ignore')

from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "/home/chungyili/.cache/huggingface/hub/models--Qwen--Qwen3.5-9B/snapshots/c202236235762e1c871ad0ccb60c8ee5ba337b9a"

tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL, trust_remote_code=True, dtype=torch.float16, device_map='cuda'
)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
weight_gb = sum(p.numel() * p.element_size() for p in model.parameters()) / 1e9
print(f"Parameters: {n_params/1e9:.1f}B")
print(f"Weight memory: {weight_gb:.1f} GB (fp16)")

prompt = "The key insight behind efficient LLM serving is"
inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
print(f"Prompt: {inputs['input_ids'].shape[1]} tokens")

Parameters: 9.0B
Weight memory: 17.9 GB (fp16)
Prompt: 9 tokens


In [ ]:
# Profile one eager forward pass
with torch.no_grad():
    model(**inputs)  # warm up
    torch.cuda.synchronize()
    with torch.profiler.profile(activities=[torch.profiler.ProfilerActivity.CUDA]) as prof:
        model(**inputs)
        torch.cuda.synchronize()

cuda_ev = [e for e in prof.events() if e.device_type == torch.autograd.DeviceType.CUDA]
print(f"CUDA kernel launches per forward: {len(cuda_ev)}")

cats = {}
for e in cuda_ev:
    n = e.name.lower()
    if any(k in n for k in ['gemm','cutlass','cublas']): cat = 'GEMM'
    elif any(k in n for k in ['elementwise','vectorized','pointwise']): cat = 'Elementwise'
    elif any(k in n for k in ['copy','memcpy','memset']): cat = 'Mem ops'
    else: cat = 'Other'
    cats[cat] = cats.get(cat, 0) + 1
print("\nKernel breakdown:")
for cat, cnt in sorted(cats.items(), key=lambda x: -x[1]):
    print(f"  {cat:15s} {cnt:5d}")

CUDA kernel launches per forward: 12921

Kernel breakdown:
  Elementwise     10520
  Other            1813
  GEMM              465
  Mem ops           123


In [ ]:
# Decode benchmark at batch 1
with torch.no_grad():
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    out = model.generate(**inputs, max_new_tokens=64, do_sample=False)
    torch.cuda.synchronize()
    gen_s = time.perf_counter() - t0
    n_new = out.shape[1] - inputs['input_ids'].shape[1]

toks = n_new / gen_s
ceiling = 1.79e12 / (weight_gb * 1e9)
print(f"Generated {n_new} tokens in {gen_s*1000:.0f} ms")
print(f"Decode throughput: {toks:.1f} tok/s")
print(f"Bandwidth ceiling (RTX 5090): {ceiling:.1f} tok/s")
print(f"Efficiency: {toks/ceiling*100:.1f}%")

Generated 64 tokens in 2086 ms
Decode throughput: 30.7 tok/s
Bandwidth ceiling (RTX 5090): 100.0 tok/s
Efficiency: 30.7%


12,921 CUDA kernels per forward pass. 10,520 of them are elementwise operations (activations, residual adds, type casts) that each do a separate HBM round-trip. The GPU reaches 30.7% of its bandwidth ceiling at batch 1.

## Section 2: TorchDynamo — capture the FX graph

`torch.compile` starts with **TorchDynamo**, which intercepts Python bytecode via CPython's `set_eval_frame` hook (PEP 523). It symbolically executes the forward pass and captures every PyTorch operation into a static **FX graph**. This graph has no Python control flow — it is a pure dataflow representation that the next stage (Inductor) can optimize.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch.fx import symbolic_trace

class TinyMLP(nn.Module):
    def __init__(self, d=256):
        super().__init__()
        self.norm = nn.RMSNorm(d)
        self.up = nn.Linear(d, d*4, bias=False)
        self.down = nn.Linear(d*4, d, bias=False)
    def forward(self, x):
        x = self.norm(x)
        x = F.silu(self.up(x))
        x = self.down(x)
        return x

traced = symbolic_trace(TinyMLP(256))
print("FX graph of TinyMLP(RMSNorm -> Linear -> SiLU -> Linear):")
print(traced.graph)

FX graph of TinyMLP(RMSNorm -> Linear -> SiLU -> Linear):
graph():
    %x : [num_users=1] = placeholder[target=x]
    %norm : [num_users=1] = call_module[target=norm](args = (%x,), kwargs = {})
    %up : [num_users=1] = call_module[target=up](args = (%norm,), kwargs = {})
    %silu : [num_users=1] = call_function[target=torch.nn.functional.silu](args = (%up,), kwargs = {inplace: False})
    %down : [num_users=1] = call_module[target=down](args = (%silu,), kwargs = {})
    return down


Each node in the FX graph is one operation. Dynamo captures this graph from the real Qwen3.5-9B model (which has thousands of nodes), then hands it to TorchInductor for optimization.

## Section 3: TorchInductor — fuse ops into Triton kernels

**TorchInductor** takes the FX graph and decides which operations to fuse. When it merges a sequence like RMSNorm → RoPE → residual add into a single Triton kernel, the intermediate values stay in GPU registers and shared memory (SRAM) instead of being written back to HBM between each operation. This eliminates HBM round-trips.

In [ ]:
# Eager forward timing
with torch.no_grad():
    for _ in range(3): model(**inputs)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(10): model(**inputs)
    torch.cuda.synchronize()
    eager_ms = (time.perf_counter() - t0) / 10 * 1000
print(f"Eager forward: {eager_ms:.1f} ms")

# Compile with Inductor
print("Compiling with torch.compile(backend='inductor')...")
compiled = torch.compile(model, backend='inductor')
with torch.no_grad():
    compiled(**inputs)  # triggers compilation
    compiled(**inputs)  # second pass
    torch.cuda.synchronize()
print("Compilation complete.")

# Profile compiled forward
with torch.no_grad():
    with torch.profiler.profile(activities=[torch.profiler.ProfilerActivity.CUDA]) as prof_c:
        compiled(**inputs)
        torch.cuda.synchronize()
compiled_kernels = len([e for e in prof_c.events()
                        if e.device_type == torch.autograd.DeviceType.CUDA])

# Benchmark
with torch.no_grad():
    for _ in range(5): compiled(**inputs)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(10): compiled(**inputs)
    torch.cuda.synchronize()
    compiled_ms = (time.perf_counter() - t0) / 10 * 1000

print(f"\nKernel launches: {len(cuda_ev)} -> {compiled_kernels} ({len(cuda_ev) - compiled_kernels} fused away)")
print(f"Forward time: {eager_ms:.1f} ms -> {compiled_ms:.1f} ms ({eager_ms/compiled_ms:.1f}x speedup)")

Eager forward: 136.4 ms
Compiling with torch.compile(backend='inductor')...
Compilation complete.

Kernel launches: 12921 -> 3379 (9542 fused away)
Forward time: 136.4 ms -> 37.7 ms (3.6x speedup)


Inductor fused 12,921 kernel launches down to 3,379 by merging 9,542 elementwise operations into Triton kernels. Each fused kernel reads its input from HBM once, performs multiple operations in SRAM, and writes the output once. The forward pass is 3.6x faster.

## Section 4: CUDA graphs — eliminate dispatch overhead

Even after fusion, each of the 3,379 remaining kernels costs ~5 μs of CPU dispatch overhead (Python → C++ → CUDA driver). CUDA graphs solve this by recording the entire kernel sequence once and replaying it with a single `cudaGraphLaunch` call.

In [ ]:
# Compile with CUDA graphs
print("Compiling with mode='reduce-overhead' (CUDA graphs)...")
compiled_cg = torch.compile(model, backend='inductor', mode='reduce-overhead')
with torch.no_grad():
    for i in range(3):
        compiled_cg(**inputs)
        torch.cuda.synchronize()

# Benchmark
with torch.no_grad():
    for _ in range(5): compiled_cg(**inputs)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(10): compiled_cg(**inputs)
    torch.cuda.synchronize()
    cg_ms = (time.perf_counter() - t0) / 10 * 1000

print(f"Forward time: {compiled_ms:.1f} ms -> {cg_ms:.1f} ms ({compiled_ms/cg_ms:.1f}x over Inductor)")
print(f"Total speedup over eager: {eager_ms/cg_ms:.1f}x")

Compiling with mode='reduce-overhead' (CUDA graphs)...
Forward time: 37.7 ms -> 16.0 ms (2.3x over Inductor)
Total speedup over eager: 8.5x


## Section 5: Summary

Each stage removes a different bottleneck:

| Stage | What it does | Bottleneck removed |
|---|---|---|
| **TorchDynamo** | Captures the FX graph | Enables the next two stages |
| **TorchInductor** | Fuses ops into Triton kernels | HBM round-trips between ops |
| **CUDA graphs** | Replays kernel sequence in one launch | CPU dispatch overhead per kernel |

In [ ]:
print(f"{'Stage':<35s} {'Kernels':>8s} {'Time (ms)':>10s} {'Speedup':>8s}")
print("-" * 65)
print(f"{'Eager (no compile)':<35s} {len(cuda_ev):>8d} {eager_ms:>10.1f} {'1.0x':>8s}")
print(f"{'torch.compile (Inductor)':<35s} {compiled_kernels:>8d} {compiled_ms:>10.1f} {f'{eager_ms/compiled_ms:.1f}x':>8s}")
print(f"{'torch.compile + CUDA graphs':<35s} {'—':>8s} {cg_ms:>10.1f} {f'{eager_ms/cg_ms:.1f}x':>8s}")

Stage                                Kernels  Time (ms)  Speedup
-----------------------------------------------------------------
Eager (no compile)                     12921      136.4     1.0x
torch.compile (Inductor)                3379       37.7     3.6x
torch.compile + CUDA graphs                —       16.0     8.5x


vLLM applies all three stages to the model it serves: TorchDynamo captures the graph, TorchInductor fuses operations, and CUDA graphs replay the fused sequence. Combined with PagedAttention (a custom CUDA kernel for paged KV cache) and continuous batching (a scheduler that amortizes weight reads across users), these optimizations explain the 4x+ throughput gap between `model.generate()` and `vllm serve`.